In [1]:
import nltk
nltk.download('punkt')
nltk.download('wordnet')
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()
import json
import pickle

import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation, Dropout
from tensorflow.keras.optimizers import SGD   
import random

words=[]
classes = []
documents = []
ignore_words = ['?', '!']
data_file = open('intents.json').read()
intents = json.loads(data_file)


for intent in intents['intents']:
    for pattern in intent['patterns']:

        # take each word and tokenize it
        w = nltk.word_tokenize(pattern)
        words.extend(w)
        # adding documents
        documents.append((w, intent['tag']))

        # adding classes to our class list
        if intent['tag'] not in classes:
            classes.append(intent['tag'])

words = [lemmatizer.lemmatize(w.lower()) for w in words if w not in ignore_words]
words = sorted(list(set(words)))

classes = sorted(list(set(classes)))

print (len(documents), "documents")

print (len(classes), "classes", classes)

print (len(words), "unique lemmatized words", words)


pickle.dump(words,open('wordsD.pkl','wb'))
pickle.dump(classes,open('classesD.pkl','wb'))

# initializing training data
training = []
output_empty = [0] * len(classes)
for doc in documents:
    # initializing bag of words
    bag = []
    # list of tokenized words for the pattern
    pattern_words = doc[0]
    # lemmatize each word - create base word, in attempt to represent related words
    pattern_words = [lemmatizer.lemmatize(word.lower()) for word in pattern_words]
    # create our bag of words array with 1, if word match found in current pattern
    for w in words:
        bag.append(1) if w in pattern_words else bag.append(0)

    # output is a '0' for each tag and '1' for current tag (for each pattern)
    output_row = list(output_empty)
    output_row[classes.index(doc[1])] = 1

    training.append([bag, output_row])
# shuffle our features and turn into np.array
random.shuffle(training)
#training = np.array(training)
# create train and test lists. X - patterns, Y - intents
train_x = [item[0] for item in training]
train_y = [item[1] for item in training]
print("Training data created")


# Create model - 3 layers. First layer 128 neurons, second layer 64 neurons and 3rd output layer contains number of neurons
# equal to number of intents to predict output intent with softmax
model = Sequential()
model.add(Dense(128, input_shape=(len(train_x[0]),), activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(len(train_y[0]), activation='softmax'))

# Compile model. Stochastic gradient descent with Nesterov accelerated gradient gives good results for this model
sgd = SGD(lr=0.01, momentum=0.9, nesterov=True)
model.compile(loss='categorical_crossentropy', optimizer=sgd, metrics=['accuracy'])

#fitting and saving the model
hist = model.fit(np.array(train_x), np.array(train_y), epochs=200, batch_size=5, verbose=1)
model.save('chatbot_modelD.h5', hist)

print("model created")


[nltk_data] Downloading package punkt to C:\Users\SPIRO-
[nltk_data]     PYTHON1\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\SPIRO-
[nltk_data]     PYTHON1\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


262 documents
46 classes ['deepfake_and_artificial_intelligence', 'deepfake_authentication', 'deepfake_benefits', 'deepfake_case_studies', 'deepfake_community_responses', 'deepfake_detection_challenges', 'deepfake_detection_future', 'deepfake_detection_reliability', 'deepfake_detection_signs', 'deepfake_detection_tools', 'deepfake_digital_forensics', 'deepfake_education', 'deepfake_ethical_concerns', 'deepfake_ethics', 'deepfake_fraud', 'deepfake_future', 'deepfake_future_trends', 'deepfake_identification', 'deepfake_impact', 'deepfake_impact_on_journalism', 'deepfake_impact_on_relationships', 'deepfake_in_entertainment', 'deepfake_industry_impact', 'deepfake_legal_aspects', 'deepfake_legal_implications', 'deepfake_media_literacy', 'deepfake_media_regulation', 'deepfake_misinformation', 'deepfake_prevention_methods', 'deepfake_prevention_technology', 'deepfake_protection_measures', 'deepfake_public_perception', 'deepfake_research', 'deepfake_security_measures', 'deepfake_social_implica

Epoch 1/200
53/53 [==============================] - 1s 3ms/step - loss: 3.8560 - accuracy: 0.0191
Epoch 2/200
53/53 [==============================] - 0s 3ms/step - loss: 3.7869 - accuracy: 0.0534
Epoch 3/200
53/53 [==============================] - 0s 3ms/step - loss: 3.7444 - accuracy: 0.0649
Epoch 4/200
53/53 [==============================] - 0s 3ms/step - loss: 3.6981 - accuracy: 0.0725
Epoch 5/200
53/53 [==============================] - 0s 2ms/step - loss: 3.6604 - accuracy: 0.0916
Epoch 6/200
53/53 [==============================] - 0s 2ms/step - loss: 3.5758 - accuracy: 0.1031
Epoch 7/200
53/53 [==============================] - 0s 3ms/step - loss: 3.4934 - accuracy: 0.1107
Epoch 8/200
53/53 [==============================] - 0s 3ms/step - loss: 3.4762 - accuracy: 0.1298
Epoch 9/200
53/53 [==============================] - 0s 3ms/step - loss: 3.3938 - accuracy: 0.1260
Epoch 10/200
53/53 [==============================] - 0s 2ms/step - loss: 3.2782 - accuracy: 0.1641
Epoch 11/

C:\Users\SPIRO-PYTHON1\AppData\Roaming\Python\Python39\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [2]:
# !pip freeze > requirements.txt

In [3]:
# !python --version

In [4]:
#! pip install -r requirements.txt
# pip install --upgrade googletrans==4.0.0-rc1
# pip install translate